In [1]:
!pip install xgboost pandas torch numpy scikit-learn statsmodels

In [15]:
import pandas as pd

df = pd.read_csv("student_career_success_dataset.csv")
print(df.head())
print(df.describe())
print(df.describe(include='object'))

TARGET = 'Company_Tier'
OUTPUTS_DIR = 'outputs/cls/'

  Student_ID  Age  Gender University_Year                   Major  \
0   ST000001   22    Male       Sophomore        Computer Science   
1   ST000002   20  Female        Freshman        Computer Science   
2   ST000003   23    Male          Senior  Information Technology   
3   ST000004   23  Female          Senior  Information Technology   
4   ST000005   23  Female          Senior    Software Engineering   

   Attendance_Percentage  Study_Hours_Per_Week  CGPA Academic_Performance  \
0                     86                    21  3.28                 Good   
1                     64                    18  2.53                 Poor   
2                     75                    19  2.83              Average   
3                     80                    19  2.80              Average   
4                     73                    19  2.93              Average   

   Programming_Skill  ...  Teamwork  Problem_Solving  English_Proficiency  \
0                  9  ...         7          

In [16]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import os
import pandas as pd

def get_numeric_columns(df: pd.DataFrame) -> list:
    return df.select_dtypes(include=[np.number]).columns.tolist()

def get_categorical_columns(df: pd.DataFrame) -> list:
    return df.select_dtypes(include=["object", "category"]).columns.tolist()


def encode_categorical(df: pd.DataFrame) -> pd.DataFrame:
    """Koduje zmienne kategorialne za pomocą Label Encoding."""
    df_encoded = df.copy()
    categorical_cols = get_categorical_columns(df)
    
    for col in categorical_cols:
        le = LabelEncoder()
        df_encoded[col] = le.fit_transform(df[col].astype(str))
    
    return df_encoded


def compute_correlation_matrix(df: pd.DataFrame, method: str = "pearson", include_categorical: bool = True) -> pd.DataFrame:
    """Oblicza macierz korelacji, opcjonalnie z uwzględnieniem zmiennych kategorialnych."""
    if include_categorical:
        df_encoded = encode_categorical(df)
        return df_encoded.corr(method=method)
    else:
        numeric_df = df.select_dtypes(include=[np.number])
        return numeric_df.corr(method=method)


def plot_correlation_matrix(
    corr_matrix: pd.DataFrame,
    output_path: str = f"{OUTPUTS_DIR}/correlation_matrix.png",
    figsize: tuple = (18, 16),
    cmap: str = "coolwarm",
    title: str = "Macierz korelacji wszystkich zmiennych"
):
    plt.figure(figsize=figsize)
    
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
    
    sns.heatmap(
        corr_matrix,
        mask=mask,
        annot=True,
        fmt=".2f",
        cmap=cmap,
        center=0,
        square=True,
        linewidths=0.5,
        cbar_kws={"shrink": 0.8},
        annot_kws={"size": 7}
    )
    
    plt.title(title, fontsize=16, fontweight="bold")
    plt.xticks(rotation=45, ha="right", fontsize=8)
    plt.yticks(fontsize=8)
    plt.tight_layout()
    
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Macierz korelacji zapisana do: {output_path}")


def get_top_correlations(corr_matrix: pd.DataFrame, n: int = 10) -> pd.DataFrame:
    corr_pairs = corr_matrix.unstack()
    
    corr_pairs = corr_pairs[corr_pairs.index.get_level_values(0) < corr_pairs.index.get_level_values(1)]
    
    corr_pairs = corr_pairs.reindex(corr_pairs.abs().sort_values(ascending=False).index)
    
    top_corr = pd.DataFrame({
        "Zmienna 1": [idx[0] for idx in corr_pairs.head(n).index],
        "Zmienna 2": [idx[1] for idx in corr_pairs.head(n).index],
        "Korelacja": corr_pairs.head(n).values
    })
    
    return top_corr

corr_matrix = compute_correlation_matrix(df, method="pearson", include_categorical=True)
plot_correlation_matrix(corr_matrix, output_path=f"{OUTPUTS_DIR}/correlation_matrix.png", title="Macierz korelacji wszystkich zmiennych")
print("Top 10 korelacji:")
print(get_top_correlations(corr_matrix, n=10))
df = df.drop(columns=['Employability_Score', 'Starting_Salary_USD', 'Student_ID', 'English_Proficiency'])

Macierz korelacji zapisana do: outputs/cls//correlation_matrix.png
Top 10 korelacji:
             Zmienna 1            Zmienna 2  Korelacja
0     Placement_Status  Starting_Salary_USD   0.949921
1         Company_Tier     Placement_Status   0.901042
2  Employability_Score      Interview_Score   0.844578
3  Employability_Score    Programming_Skill   0.814310
4  Employability_Score         Resume_Score   0.790519
5  Employability_Score          Internships   0.789657
6         Company_Tier  Starting_Salary_USD   0.781823
7  Employability_Score      Problem_Solving   0.764074
8  Employability_Score   Projects_Completed   0.752465
9      Problem_Solving    Programming_Skill   0.736964


In [4]:
from statsmodels.formula.api import ols
import statsmodels.api as sm

symbol = '+'

# Q("nazwa_zmiennej") - syntax umożliwiający posługiwanie się pełnymi nazwami kolumn do zdefiniowania modelu liniowego
# C(nazwa_zmiennej) - wskazanie, że dana zmienna jest zmienną kategorialną (jakościową)

categorical_vars = "".join([f'C(Q("{var}")) {symbol} ' for var in get_categorical_columns(df) if var != TARGET])
numeric_vars = "".join([f'{symbol if var != get_numeric_columns(df)[0] else ""} Q("{var}") ' for var in get_numeric_columns(df) if var != TARGET])

definition = f'Q("{TARGET}") ~ ' + categorical_vars + numeric_vars
print(definition)
print(df.columns)
stats_model = ols(definition, data=df).fit()
anova_result = sm.stats.anova_lm(stats_model, type=2)
print(anova_result)

"""
Najistotniejszą zmienną jest Cholesterol, Hemisphere, Diabetes, Alcohol Consumption i Sleep Hours Per Day, 
jednakże ŻADNA nie jest statystycznie istotna, bo ich PR(>F) jest wyższe od 0.05. 
Wyżej wymienione zmienne to top 5. Skoro nie można zbudować na ich podstawie modelu liniowego, 
to należy albo odnaleźć nieliniowe zależności, albo posłużyć się głęboką siecią neuronową, 
która w kolejnych etapach będzie tworzyć coraz istotniejsze serie danych dla zmiennej zależnej.
"""


Q("Company_Tier") ~ C(Q("Gender")) + C(Q("University_Year")) + C(Q("Major")) + C(Q("Academic_Performance")) + C(Q("GitHub_Profile")) + C(Q("Leadership_Experience")) + C(Q("LinkedIn_Profile")) + C(Q("English_Proficiency")) + C(Q("Career_Field")) + C(Q("Placement_Mode")) +  Q("Age") + Q("Attendance_Percentage") + Q("Study_Hours_Per_Week") + Q("CGPA") + Q("Programming_Skill") + Q("Projects_Completed") + Q("Certifications") + Q("Hackathons") + Q("Internships") + Q("Resume_Score") + Q("Communication_Skills") + Q("Teamwork") + Q("Problem_Solving") + Q("Interview_Score") + Q("Starting_Salary_USD") 
Index(['Age', 'Gender', 'University_Year', 'Major', 'Attendance_Percentage',
       'Study_Hours_Per_Week', 'CGPA', 'Academic_Performance',
       'Programming_Skill', 'Projects_Completed', 'Certifications',
       'Hackathons', 'GitHub_Profile', 'Internships', 'Leadership_Experience',
       'LinkedIn_Profile', 'Resume_Score', 'Communication_Skills', 'Teamwork',
       'Problem_Solving', 'English_

ValueError: endog has evaluated to an array with multiple columns that has shape (50000, 4). This occurs when the variable converted to endog is non-numeric (e.g., bool or str).

In [ ]:
def plot_scatter_plot(df, x, y):
    plt.scatter(df[x], df[y])
    plt.xlabel(x)
    plt.ylabel(y)
    plt.title(f"{x} & {y}")
    os.makedirs(f"{OUTPUTS_DIR}/scatterplots/", exist_ok=True)
    plt.savefig(f"{OUTPUTS_DIR}/scatterplots/{x} & {y}.png")
    plt.close()

for i in range(len(df.columns)):
    plot_scatter_plot(df, df.columns[i], TARGET)
    # for j in range(i + 1, len(df.columns)):
    #     plot_scatter_plot(df, df.columns[i], df.columns[j])

"""
Niestety, analiza wykresów punktowych nie wykazała żadnych zależności pomiędzy zmienną zależną, a pozostałymi
"""

In [17]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

def create_preprocessor(df: pd.DataFrame, target_col: str):
    categorical = df.select_dtypes(include=["object"]).columns.tolist()

    if target_col in categorical:
        categorical.remove(target_col)

    for g in ["passed"]:
        if g in categorical:
            categorical.remove(g)

    numeric = df.select_dtypes(exclude=["object"]).columns.tolist()


    if target_col in numeric:
        numeric.remove(target_col)

    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
            ("num", StandardScaler(), numeric),
        ]
    )
    return preprocessor

RANDOM_STATE = 42
TEST_SIZE = int(df.__len__() * 0.4)

X = df.drop(columns=[TARGET])
y = df[TARGET]


train_x, test_x, train_y, test_y = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y,
)
print()

preprocessor = create_preprocessor(df, TARGET)

train_x_t = preprocessor.fit_transform(train_x)
test_x_t = preprocessor.transform(test_x)


le = LabelEncoder()
train_y_t = le.fit_transform(train_y)
test_y_t = le.transform(test_y)



In [18]:
import torch

torch.manual_seed(RANDOM_STATE)
x_train_t = torch.tensor(train_x_t.toarray() if hasattr(train_x_t, "toarray") else train_x_t, dtype=torch.float32)
x_test_t = torch.tensor(test_x_t.toarray() if hasattr(test_x_t, "toarray") else test_x_t, dtype=torch.float32)


y_train_cls_t = torch.tensor(train_y_t, dtype=torch.long)
y_test_cls_t = torch.tensor(test_y_t, dtype=torch.long)

print(x_train_t.shape)



torch.Size([30000, 63])


In [19]:

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import accuracy_score, classification_report

nb_model = BernoulliNB(alpha=1.0)

nb_model.fit(train_x_t, train_y_t)
y_pred = nb_model.predict(test_x_t)
acc = accuracy_score(test_y_t, y_pred)
report = classification_report(test_y_t, y_pred, output_dict=True)

print("NB Model - Accuracy: ", acc)
print(report)

lr_model = LogisticRegression(random_state=42)
lr_model.fit(train_x_t, train_y_t)
y_pred = lr_model.predict(test_x_t)

acc = accuracy_score(test_y_t, y_pred)
report = classification_report(test_y_t, y_pred, output_dict=True)

print("LR Model - Accuracy: ", acc)
print(report)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(train_x_t, train_y_t)
y_pred = rf_model.predict(test_x_t)

acc = accuracy_score(test_y_t, y_pred)
report = classification_report(test_y_t, y_pred, output_dict=True)

print("RF Model - Accuracy: ", acc)
print(report)

xgboost_model = XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8,
        random_state=RANDOM_STATE,
        eval_metric="mlogloss"
)
xgboost_model.fit(train_x_t, train_y_t)

y_pred = xgboost_model.predict(test_x_t)

acc = accuracy_score(test_y_t, y_pred)
report = classification_report(test_y_t, y_pred, output_dict=True)


print("XGB Model - Accuracy: ", acc)
print(report)



XGBoostError: 
XGBoost Library (xgboost.dll) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ['[WinError 4551] Zasady kontroli aplikacji zablokowały ten plik']


In [20]:
from sklearn.metrics import confusion_matrix, roc_curve, auc, classification_report

def plot_roc_auc(y_test, y_score, output_path, model_name="Model"):
    fpr, tpr, _ = roc_curve(y_test, y_score)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f'{model_name} (AUC = {roc_auc:.2f})')

    plt.plot([0, 1], [0, 1], 'r--', label='Random Guess')

    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curves for Model')
    plt.legend()
    plt.savefig(output_path + " roc auc curve.png")
    plt.close()
    return roc_auc

def normalize_confusion_matrix(cm, norm='true'):
    """
    Normalize a confusion matrix.
    
    Parameters:
    cm (array-like): Confusion matrix to be normalized.
    norm (str): Type of normalization ('true', 'pred', 'all').
    
    Returns:
    ndarray: Normalized confusion matrix.
    """
    if norm == 'true':
        cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    elif norm == 'pred':
        cm_normalized = cm.astype('float') / cm.sum(axis=0)[np.newaxis, :]
    elif norm == 'all':
        cm_normalized = cm.astype('float') / cm.sum()
    else:
        raise ValueError("Unknown normalization type. Use 'true', 'pred', or 'all'.")
    
    return cm_normalized

def plot_confusion_matrix(y_test, y_pred, output_dir):
    cm = confusion_matrix(y_test, y_pred)
    normalized = normalize_confusion_matrix(cm, norm='true')
    plt.figure(figsize=(5, 4))
    sns.heatmap(normalized, annot=True, fmt=".2f", cmap="Blues")
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    os.makedirs(os.path.dirname(output_dir), exist_ok=True)
    plt.savefig(output_dir+" confusion matrix.png")
    plt.close()

def plot_training(total_loss, total_acc, epochs, output_path):
    fig, ax = plt.subplots()
    
    ax.plot(epochs, total_loss, color='lightblue', linewidth=3)
    ax.plot(epochs, total_acc, color="red", linewidth=4, marker="o")
    ax.set(xlabel="epochs", ylabel="loss/accuracy")
    
    plt.savefig(output_path+" training.png")
    plt.close()


In [ ]:
from sklearn.metrics import classification_report
import torch

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score


def get_class_weights(y_t):
    counts = torch.bincount(y_t)
    weights = counts.sum() / (len(counts) * counts.float().clamp_min(1))
    return weights

def train_model_full_batch(model, x_train_t, y_train_t, epochs=200, lr=0.001,
                           weight_decay=1e-5, logging_step=10, l1_param=0,
                           output_path=f"{OUTPUTS_DIR}/full-batch"):
    criterion = nn.CrossEntropyLoss(weight=get_class_weights(y_train_t))
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    losses = []
    accuracies = []
    all_epochs = []

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        logits = model(x_train_t)
        loss = criterion(logits, y_train_t)
        total_loss = loss + sum(param.abs().sum() for param in model.parameters()) * l1_param

        total_loss.backward()
        optimizer.step()

        if logging_step != -1 and epoch % logging_step == logging_step - 1:
            preds = logits.argmax(dim=1)
            acc = (preds == y_train_t).float().mean().item()

            print(f"Full batch | Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}, Accuracy: {acc:.4f}")

            losses.append(total_loss.item())
            accuracies.append(acc)
            all_epochs.append(epoch + 1)
            plot_training(losses, accuracies, all_epochs, output_path)

    return losses, accuracies


def train_model_mini_batch(model, x_train_t, y_train_t, epochs=200, batch_size=128,
                           lr=0.001, weight_decay=1e-5, logging_step=10,
                           l1_param=0, output_path=f"{OUTPUTS_DIR}/mini-batch"):
    dataset = TensorDataset(x_train_t, y_train_t)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    criterion = nn.CrossEntropyLoss(weight=get_class_weights(y_train_t))
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    losses = []
    accuracies = []
    all_epochs = []

    for epoch in range(epochs):
        model.train()

        epoch_loss = 0.0
        correct = 0
        total = 0

        for batch_x, batch_y in loader:
            optimizer.zero_grad()

            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            total_loss = loss + sum(param.abs().sum() for param in model.parameters()) * l1_param

            total_loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * batch_y.size(0)
            correct += (logits.argmax(dim=1) == batch_y).sum().item()
            total += batch_y.size(0)

        if logging_step != -1 and epoch % logging_step == logging_step - 1:
            avg_loss = epoch_loss / total
            acc = correct / total

            print(f"Mini batch | Epoch [{epoch + 1}/{epochs}], Loss: {avg_loss:.4f}, Accuracy: {acc:.4f}")

            losses.append(avg_loss)
            accuracies.append(acc)
            all_epochs.append(epoch + 1)
            plot_training(losses, accuracies, all_epochs, output_path)

    return losses, accuracies


def evaluate_model_multiclass(model, x_test_t, y_test_t, le, output_dir, log=True):
    model.eval()

    with torch.no_grad():
        logits = model(x_test_t)
        probs = torch.softmax(logits, dim=1).cpu().numpy()

        y_true = y_test_t.cpu().numpy()
        y_pred = logits.argmax(dim=1).cpu().numpy()

        y_true_labels = le.inverse_transform(y_true)
        y_pred_labels = le.inverse_transform(y_pred)

        acc = accuracy_score(y_true_labels, y_pred_labels)
        report = classification_report(y_true_labels, y_pred_labels, target_names=le.classes_)

        try:
            roc_auc = roc_auc_score(y_true, probs, multi_class="ovr", average="macro")
        except ValueError:
            roc_auc = np.nan

        if log:
            print(f"Accuracy: {acc:.4f}")
            print(report)
            print(f"Macro ROC AUC OvR: {roc_auc:.4f}")

            plot_confusion_matrix(y_true_labels, y_pred_labels, output_dir)

    return acc


In [9]:
from datetime import datetime

import torch.nn as nn
class DeepNet(nn.Module):
    def __init__(self, input: int = 26, hidden_layer: int = 26 * 4, output: int = 4, dropout=0.1):
        super(DeepNet, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input, hidden_layer),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_layer, output)
        )

    def forward(self, x):
        return self.net(x)

In [21]:

from copy import deepcopy

full_batch_model = DeepNet(
    input=x_train_t.shape[1],
    hidden_layer=128,
    output=len(le.classes_),
    dropout=0.2
)
mini_batch_model = DeepNet(
    input=x_train_t.shape[1],
    hidden_layer=128,
    output=len(le.classes_),
    dropout=0.2
)

timestamp = datetime.now().strftime("%H-%M-%S")
full_batch_output_path = f"{OUTPUTS_DIR}/{timestamp}-full-batch"
mini_batch_output_path = f"{OUTPUTS_DIR}/{timestamp}-mini-batch"

train_model_mini_batch(
    mini_batch_model,
    x_train_t,
    y_train_cls_t,
    epochs=20,
    batch_size=128,
    lr=0.001,
    weight_decay=1e-5,
    logging_step=1,
    l1_param=0,
    output_path=mini_batch_output_path
)
train_model_full_batch(
    full_batch_model,
    x_train_t,
    y_train_cls_t,
    epochs=20,
    lr=0.001,
    weight_decay=1e-5,
    logging_step=5,
    l1_param=0,
    output_path=full_batch_output_path
)


print("Full-batch evaluation")
full_batch_acc = evaluate_model_multiclass(
    full_batch_model,
    x_test_t,
    y_test_cls_t,
    le,
    full_batch_output_path
)

print("Mini-batch evaluation")
mini_batch_acc = evaluate_model_multiclass(
    mini_batch_model,
    x_test_t,
    y_test_cls_t,
    le,
    mini_batch_output_path
)

print(f"Full-batch accuracy: {full_batch_acc:.4f}")
print(f"Mini-batch accuracy: {mini_batch_acc:.4f}")
    

Mini batch | Epoch [1/20], Loss: 0.5130, Accuracy: 0.9087
Mini batch | Epoch [2/20], Loss: 0.1570, Accuracy: 0.9696
Mini batch | Epoch [3/20], Loss: 0.1086, Accuracy: 0.9754
Mini batch | Epoch [4/20], Loss: 0.0826, Accuracy: 0.9792
Mini batch | Epoch [5/20], Loss: 0.0637, Accuracy: 0.9824
Mini batch | Epoch [6/20], Loss: 0.0575, Accuracy: 0.9840
Mini batch | Epoch [7/20], Loss: 0.0495, Accuracy: 0.9865
Mini batch | Epoch [8/20], Loss: 0.0413, Accuracy: 0.9881
Mini batch | Epoch [9/20], Loss: 0.0405, Accuracy: 0.9884
Mini batch | Epoch [10/20], Loss: 0.0366, Accuracy: 0.9892
Mini batch | Epoch [11/20], Loss: 0.0365, Accuracy: 0.9901
Mini batch | Epoch [12/20], Loss: 0.0308, Accuracy: 0.9910
Mini batch | Epoch [13/20], Loss: 0.0241, Accuracy: 0.9919
Mini batch | Epoch [14/20], Loss: 0.0267, Accuracy: 0.9914
Mini batch | Epoch [15/20], Loss: 0.0247, Accuracy: 0.9922
Mini batch | Epoch [16/20], Loss: 0.0208, Accuracy: 0.9926
Mini batch | Epoch [17/20], Loss: 0.0210, Accuracy: 0.9929
Mini b

In [ ]:
import torch
import torch.nn as nn

class DeeperNet(nn.Module):
    def __init__(self, input: int = 26, hidden_layer: int = 26*4, count_of_layers=3):
        super(DeeperNet, self).__init__()
        dense_net = [[nn.Linear(hidden_layer, hidden_layer), nn.ReLU(), nn.Dropout(0.1)] for i in range(count_of_layers - 1)]
        dense_net = [sublayer for layer in dense_net for sublayer in layer]
        self.net = nn.Sequential(
            nn.Linear(input, hidden_layer),
            nn.ReLU(),
            *dense_net,
            nn.Linear(hidden_layer, 4)
        )
        

    def forward(self, x):
        x = self.net(x).argmax(dim=-1).unsqueeze(dim=-1).float()
        return x

output_path = f"{OUTPUTS_DIR}/{(datetime.now().strftime("%H-%M-%S"))}"
print(x_train_t.shape[1])
model = DeeperNet(x_train_t.shape[1], hidden_layer=256, count_of_layers=10)
train_model(model, x_train_t, y_train_t, epochs=50, lr=0.001, weight_decay=1e-5, logging_step = 5, output_path=output_path, l1_param=0)
evaluate_model(model, x_test_t, y_test_t, le, output_path)

In [ ]:
import optuna

def objective(trial):
    hidden_size = trial.suggest_int("hidden_size", 10, 1000)
    dropout = trial.suggest_float("dropout", 0, 1)
    model = DeepNet(x_train_t.shape[1], hidden_layer=hidden_size, dropout=dropout)
    train_model(model, x_train_t, y_train_t, epochs=100, l1_param=0, logging_step=-1)
    return evaluate_model(model, x_test_t, y_test_t, le, output_path, log = False)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=1000)
print(study.best_params)